In [ ]:
import pandas as pd
import json
import glob
import os

# --- CONFIGURATION ---
input_folder = "./csv/"  # Folder where your 1000 CSVs are
output_file = "qwen_finetune_data.jsonl"

# Keywords to identify the "User" side (Authority Figures)
# If a speaker label contains these, they are the USER.
AUTHORITY_KEYWORDS = [
    "Deputy", "Officer", "Dispatcher", "Law enforcement", 
    "Property owner", "Sergeant", "Lieutenant", "Trooper", "Police"
]

# The System Prompt acts as the "Scenario Setup"
DEFAULT_SYSTEM_PROMPT = (
    "You are a suspect in a police interaction. You are currently agitated, "
    "non-compliant, and refusing to follow orders. You believe you are being "
    "treated unfairly. Respond to the officer's commands with resistance."
)

def identify_role(speaker_label):
    """
    Decides if the speaker is the User (Officer) or Assistant (Suspect).
    """
    if pd.isna(speaker_label):
        return "user" # Default to user if unknown
        
    label_lower = str(speaker_label).lower()
    
    # Check if speaker is an authority figure
    for keyword in AUTHORITY_KEYWORDS:
        if keyword.lower() in label_lower:
            return "user"
    
    # If not authority, assume it's the Suspect (Assistant)
    return "assistant"

def process_single_csv(file_path):
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Skipping {file_path}: {e}")
        return None

    # Ensure columns exist
    required_cols = ['speaker_label', 'transcript']
    if not all(col in df.columns for col in required_cols):
        return None

    messages = [
        {"role": "system", "content": DEFAULT_SYSTEM_PROMPT}
    ]
    
    current_role = None
    current_content = []

    for _, row in df.iterrows():
        text = str(row['transcript']).strip()
        speaker = row['speaker_label']
        
        if not text or text == "nan":
            continue

        # Determine Role
        role = identify_role(speaker)

        # Merge consecutive messages from same role
        if role == current_role:
            current_content.append(text)
        else:
            # Save previous turn
            if current_role is not None:
                messages.append({
                    "role": current_role,
                    "content": " ".join(current_content)
                })
            # Start new turn
            current_role = role
            current_content = [text]

    # Append final turn
    if current_role and current_content:
        messages.append({
            "role": current_role, 
            "content": " ".join(current_content)
        })

    return {"messages": messages}

# --- MAIN EXECUTION LOOP ---
all_conversations = []

# Get all CSV files
csv_files = glob.glob(os.path.join(input_folder, "*.csv"))
print(f"Found {len(csv_files)} files. Processing...")

for file_path in csv_files:
    data = process_single_csv(file_path)
    if data:
        # Quality Control: Only keep conversations with at least 4 turns
        if len(data["messages"]) >= 4:
            all_conversations.append(data)

# Save to JSONL for Unsloth/Qwen
with open(output_file, "w", encoding="utf-8") as f:
    for conversation in all_conversations:
        f.write(json.dumps(conversation) + "\n")

print(f"Successfully processed {len(all_conversations)} conversations.")

Found 901 files. Processing...
Successfully processed 520 conversations.


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/mdhasebulhasan/Documents/Research/generative-ai/gemini/use-cases/video-analysis/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/mdhasebulhasan/Documents/Research/generative-ai/gemini/use-cases/video-analysis/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 577, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/mdhasebulhasan/Documents/Research/generative-ai/gemini/use-cases/video-analysis/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 994, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/mdhasebulhasan/Documents/Rese